# MiniGPT learns from a teacher: follow along

This notebook goes with my post [MiniGPT (Part 6)](https://haddley.github.io/posts/minigpt5/). It looks at a real teacher's wheel of chances, works out how different two wheels are, and trains a small MiniGPT with and without GPT-2 as its teacher.

**It needs a Mac with Apple Silicon**, because MLX only runs there. Open it in Jupyter from a clone of [minigpt-series](https://github.com/Haddley/minigpt-series), with the packages from its `requirements.txt` installed. The first run downloads GPT-2 (about 500 MB) from Hugging Face.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("."))
from notebook_setup import setup
run = setup("part6-distillation")

## 1. The answer, and the teacher's wheel

After "Tim gave his dog a", the guessing game only knows the one right answer. GPT-2, as a teacher, has a chance for every one of its 50,257 pieces.

In [ ]:
import mlx.core as mx
import mlx.nn as nn
from mlx_lm import load

teacher, tokenizer = load("openai-community/gpt2")
ids = tokenizer.encode("Tim gave his dog a")
logits = teacher(mx.array([ids]))[0, -1]
wheel = mx.softmax(logits)
top = mx.argsort(-wheel)[:8].tolist()
print("the teacher's wheel, biggest slices:")
for i in top:
    print(f"  {tokenizer.decode([i])!r:12} {wheel[i].item():6.1%}")
print("\nthe one-hot answer, if the story said ' bone':")
print(f"  {' bone'!r:12} 100.0%, and every other piece 0%")

## 2. How different are two wheels?

KL divergence is 0 for identical wheels, and grows as they differ. Softening both with a temperature of 2 first, as the post does, makes the near misses count.

In [ ]:
def kl(teacher_logits, student_logits, temp=2.0):
    t = nn.log_softmax(teacher_logits / temp)
    s = nn.log_softmax(student_logits / temp)
    return (mx.exp(t) * (t - s)).sum().item()

even = mx.zeros_like(logits)                  # a student who knows nothing: an even wheel
print(f"teacher against itself:      {kl(logits, logits):.4f}")
print(f"teacher against an even wheel: {kl(logits, even):.4f}")
print(f"softened wheel's top slice: {mx.softmax(logits / 2)[top[0]].item():.1%}, against {wheel[top[0]].item():.1%} unsoftened")

## 3. Train with and without a teacher

The same 30-million-number student, 3,000 steps each, on Part 3's stories with GPT-2's pieces. On my Mac Studio, 3,000 steps took about 13 minutes without a teacher, and 25 minutes with GPT-2 as the teacher.

In [ ]:
run("train_distill.py", "--tag", "baseline", "--teacher", "none", "--alpha", "1.0", "--iters", "3000")
run("train_distill.py", "--tag", "gpt2", "--teacher", "gpt2", "--alpha", "0.5", "--iters", "3000")

In [ ]:
import json
for tag in ("baseline", "gpt2"):
    h = json.load(open(f"runs/history_{tag}.json"))
    best = min(h["history"], key=lambda p: p["val"])
    print(f"{tag:9} best bits per byte {best['bpb']:.4f}, peak memory {h['peak_gb']:.1f} GB, {h['elapsed_sec'] / 60:.1f} minutes")